# บทที่ 11: พื้นฐาน SQL สำหรับเชื่อมข้อมูลเพื่อการวิเคราะห์

ในการวิเคราะห์ข้อมูลจริง ข้อมูลที่ต้องใช้มักไม่ได้อยู่ในตารางเดียว 
แต่อาจกระจายอยู่หลายชุดข้อมูล เช่น ข้อมูลจากหลายหน่วยงาน หลายระบบ หรือหลายไฟล์ 
ในบทนี้ เราจะเรียนรู้การใช้ SQL เบื้องต้นผ่าน DuckDB บน Python 
เพื่อเชื่อมข้อมูลหลายชุดเข้าด้วยกัน และสร้างผลลัพธ์ที่นำไปใช้วิเคราะห์ต่อได้

## ผลการเรียนรู้ที่คาดหวัง

เมื่อจบบทเรียนนี้ ผู้เรียนจะสามารถ: 
1. นำเข้าข้อมูลจากไฟล์ Excel และ CSV มาเป็น pandas DataFrame ได้ 
2. ใช้ DuckDB เพื่อ query ข้อมูลจาก pandas DataFrame ได้ 
3. เขียน SQL เบื้องต้นได้ เช่น `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY` 
4. ใช้ aggregate function เช่น `COUNT`, `COUNT DISTINCT`, `SUM`, `AVG`, `MIN`, `MAX` 
5. อธิบายความแตกต่างของ `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, และ `FULL OUTER JOIN` ได้ 
6. เลือกชนิดของ JOIN ให้เหมาะกับคำถามการวิเคราะห์ได้ 
7. เชื่อมข้อมูลจากหลายแหล่งในระดับพื้นที่ เช่น จังหวัด อำเภอ ตำบล ได้ 
8. ตรวจสอบผลลัพธ์หลัง JOIN ได้ เช่น ข้อมูลที่เชื่อมเจอ เชื่อมไม่เจอ หรือเกิดค่า NULL 
9. ใช้ `WITH` หรือ CTE เพื่อจัด query ให้อ่านง่ายขึ้น 
10. ห่อ SQL query เป็น Python function เพื่อเรียกใช้งานซ้ำได้

## ลำดับเนื้อหา 
บทนี้ประกอบด้วยหัวข้อดังนี้
1. แนวคิดการใช้ SQL เพื่อเชื่อมข้อมูล 
2. การนำเข้าข้อมูลตัวอย่าง 
3. การเตรียม DuckDB และ register DataFrame 
4. SQL พื้นฐาน: `SELECT`, `WHERE`, `ORDER BY` 
5. การสรุปข้อมูลด้วย `GROUP BY` 
6. การใช้ aggregate function 
7. แนวคิดของ JOIN และผลลัพธ์ที่เกิดจาก JOIN แต่ละแบบ 
8. การเชื่อมข้อมูลระดับพื้นที่ 
9. การใช้ `COALESCE` เพื่อจัดการค่า NULL หลัง JOIN 
10. การตรวจสอบผลลัพธ์หลัง JOIN 
11. การห่อ query เป็น Python function 
12. แบบฝึกหัดท้ายบท

## 3.1 แนวคิดการใช้ SQL เพื่อเชื่อมข้อมูล

SQL เป็นภาษาที่ใช้ทำงานกับข้อมูลในรูปแบบตาราง 

ในงานวิเคราะห์ เรามักใช้ SQL เพื่อตอบคำถาม เช่น 
- ข้อมูลแต่ละชุดมีจำนวน record เท่าไร 
- จังหวัดใดมีจำนวนเกษตรกรเปราะบางมากที่สุด 
- จังหวัดใดมีจำนวน logbook จาก MSO มากที่สุด 
- พื้นที่ใดมีข้อมูลจากทั้งสองแหล่ง 
- พื้นที่ใดมีข้อมูลเกษตรกรจำนวนมาก แต่ยังไม่มีข้อมูลจาก MSO 
- พื้นที่ใดควรนำไปวิเคราะห์เชิงลึกต่อ 

ในบทนี้ เราจะใช้ SQL ผ่าน DuckDB โดยให้ DuckDB query ข้อมูลที่อยู่ใน pandas DataFrame 

## 3.2 เตรียม Library

In [2]:
%pip install duckdb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd 
import duckdb

## 3.3 นำเข้าข้อมูลตัวอย่าง 
ในบทนี้จะนำเข้าข้อมูล 2 ชุด 
1. ข้อมูลเกษตรกรเปราะบางจากไฟล์ Excel 
2. ข้อมูล logbook จากไฟล์ CSV 

หลังจากอ่านไฟล์แล้ว เราจะเก็บข้อมูลไว้ใน DataFrame ชื่อ 
- `farmer_df` 
- `mso_df`

In [4]:
farmer_path = "moac_opsmoac_fragile_farmer.xlsx" 
mso_path = "msdhs_ops_mso_logbook.csv"

In [5]:
# farmer_df = pd.read_excel(farmer_path) 
# farmer_df.head()

อย่างไรก็ตาม ในการใช้งานจริง ไฟล์ข้อมูลอาจไม่ได้อยู่ใน folder เดียวกับ notebook เสมอไป ดังนั้นก่อนอ่านไฟล์ ควรตรวจสอบตำแหน่งปัจจุบันของ notebook และตรวจสอบว่าไฟล์ข้อมูลอยู่ที่ path ใด หากระบุ path ไม่ถูกต้อง เช่น ใช้แค่ชื่อไฟล์โดยตรง แต่ไฟล์อยู่คนละ directory จะเกิด error เช่น

```text 
FileNotFoundError: [Errno 2] No such file or directory
```

ดังนั้นขั้นตอนที่ดีคือ
1. ตรวจสอบ current working directory
2. กำหนด folder ที่เก็บข้อมูล
3. ประกอบ path ของไฟล์ด้วย pathlib.Path
4. ตรวจสอบว่าไฟล์มีอยู่จริงก่อนอ่าน

In [6]:
from pathlib import Path

ตรวจสอบว่า notebook กำลังทำงานอยู่ที่ directory ใด

In [7]:
current_dir = Path.cwd() 
current_dir

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day5')

กำหนด directory ที่เก็บข้อมูล

ตัวอย่างนี้ไฟล์ข้อมูลอยู่ไม่ได้อยู่ใน folder เดียวกับ notebook

โครงสร้าง folder ตัวอย่าง:

```text
course/
├── day5/
│   └── basic_sql.ipynb
└── day3/
    ├── moac_opsmoac_fragile_farmer.xlsx
    └── msdhs_ops_mso_logbook.csv
```

ถ้า notebook อยู่ใน folder notebooks และข้อมูลอยู่ใน folder data ที่อยู่ข้างนอก notebook folder
จะต้องใช้ path `/workspaces/MSDHS_OJT/py-jupyter_docker/course`

In [8]:
data_dir = Path("/workspaces/MSDHS_OJT/py-jupyter_docker/course") 
data_dir

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course')

กำหนด path ของไฟล์ข้อมูลแต่ละไฟล์

In [9]:
farmer_path = data_dir / "day3/moac_opsmoac_fragile_farmer.xlsx" 
mso_path = data_dir / "day3/msdhs_ops_mso_logbook.csv" 

In [10]:
farmer_path

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3/moac_opsmoac_fragile_farmer.xlsx')

In [11]:
mso_path

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3/msdhs_ops_mso_logbook.csv')

ตรวจสอบว่าไฟล์มีอยู่จริงหรือไม่ก่อนอ่านข้อมูล 
ถ้าผลลัพธ์เป็น `False` แปลว่า path ยังไม่ถูกต้อง ต้องกลับไปตรวจสอบว่า 
- notebook อยู่ที่ directory ใด 
- ไฟล์ข้อมูลอยู่ที่ directory ใด 
- ชื่อไฟล์สะกดถูกต้องหรือไม่ 
- นามสกุลไฟล์ถูกต้องหรือไม่ เช่น `.xlsx`, `.csv`

In [12]:
print("Farmer file exists:", farmer_path.exists())

Farmer file exists: True


In [13]:
print("MSO file exists:", mso_path.exists())

MSO file exists: True


ถ้าต้องการดูไฟล์ทั้งหมดใน folder ข้อมูล สามารถใช้คำสั่งนี้เพื่อตรวจสอบชื่อไฟล์

In [14]:
list(data_dir.iterdir())

[PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3'),
 PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day5'),
 PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day2')]

หลังจากตรวจสอบแล้วว่า path ถูกต้อง จึงอ่านไฟล์เข้ามาเป็น DataFrame

In [15]:
mso_usecols = [ 
    "วันที่แก้ไขข้อมูลล่าสุด", 
    "รหัสครัวเรือน", 
    "รหัสประจำบ้าน", 
    "วันที่สร้างครัวเรือน",
    "ตำบล/แขวง",
    "เขต/อำเภอ/เทศบาล", 
    "จังหวัด", 
    "อายุ", 
    "เพศ", 
    "ระดับการศึกษา", 
    "อาชีพหลัก", 
    "ประเภทกลุ่มเป้าหมาย", 
    "รหัส cm", 
    "หน่วยงานของ cm" 
] 

mso_dtype = { 
    "รหัสครัวเรือน": "string", 
    "รหัสประจำบ้าน": "string", 
    "รหัส cm": "string" } 
    
mso_df = pd.read_csv( 
    mso_path, 
    encoding="utf-8-sig", 
    usecols=mso_usecols, 
    dtype=mso_dtype, 
    parse_dates=["วันที่แก้ไขข้อมูลล่าสุด", "วันที่สร้างครัวเรือน"] 
) 

mso_df

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,ตำบล/แขวง,เขต/อำเภอ/เทศบาล,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2025-07-29 14:33:32.178,688878eaa933a405b21cc136,<NA>,2025-07-29 14:31:54.610,59,NaN,โนนกอก,เกษตรสมบูรณ์,ชัยภูมิ,NaN,เกษตรกรรม (พืช ปศุสัตว์ ประมง),วัยผู้ใหญ่/วัยแรงงาน,cm360060,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.ชัยภูมิ
96,2023-03-17 19:08:15.489,628b008c8fa67bf5ad7d221b,<NA>,2022-05-23 10:33:32.270,79,NaN,เมืองเตา,พยัคฆภูมิพิสัย,มหาสารคาม,NaN,NaN,ผู้สูงอายุ,cm440005,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
97,2023-05-01 09:19:26.395,63b6325b2617b2b7aa04dc0b,25120251099,2023-05-01 09:13:47.520,51,หญิง,หนองหมากฝ้าย,วัฒนานคร,สระแก้ว,ประถมศึกษา,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm270001,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
98,2025-01-04 14:05:25.955,63e1d0d92617b2b7aa06096d,40160104483,2023-07-02 11:17:29.369,81,ชาย,หนองกุงธนสาร,ภูเวียง,ขอนแก่น,ไม่ได้เรียนหนังสือ,NaN,ผู้สูงอายุ,cm400016,ศูนย์เรียนรู้การพัฒนาสตรีและครอบครัวรัตนาภา จั...


In [16]:
target_sheets = [ 
    "v_cpd_fragile", 
    "v_dld_fragile", 
    "v_doae_fragile" 
] 

farmer_usecols = [ 
    "department_code", 
    "department", 
    "pid", 
    "province_code", 
    "province", 
    "amphur", 
    "tambon", 
    "is_farmer", 
    "farmer_type", 
    "main_occupation", 
    "income_in", 
    "income_out", 
    "debts_in", 
    "debts_out", 
    "updated_at" 
] 

farmer_dtype = { 
    "pid": "string", 
    "province_code": "string" 
} 

farmer_df_list = [] 

for sheet in target_sheets: 
    temp_df = pd.read_excel( 
        farmer_path, 
        sheet_name=sheet, 
        usecols=farmer_usecols, 
        dtype=farmer_dtype 
    ) 
    
    temp_df["source_sheet"] = sheet 
    
    farmer_df_list.append(temp_df) 
    
farmer_raw_df = pd.concat(farmer_df_list, ignore_index=True) 

farmer_raw_df

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,doae,กรมส่งเสริมการเกษตร,b437149cc5b6a79f8866914fd84b644196ed99ba4f3d62...,96,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,200000.0,0.0,0.0,0.0,20250825.0,v_doae_fragile
91,doae,กรมส่งเสริมการเกษตร,b1ce3ba83cfcfc5c21414fe2c496a37b635f56cb4de98c...,61,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,0.0,0.0,0.0,0.0,20250825.0,v_doae_fragile
92,doae,กรมส่งเสริมการเกษตร,5bdabb8ac705d3ba6fab31c9f7f1a48c37455ac548d72a...,53,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,200000.0,0.0,100000.0,0.0,20250825.0,v_doae_fragile
93,doae,กรมส่งเสริมการเกษตร,d066b479f17041cb1297e115b3ed0c8e8b724769855a97...,12,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,60000.0,0.0,0.0,0.0,20250825.0,v_doae_fragile


## 3.4 ตรวจสอบข้อมูลหลังนำเข้า 
หลังจากนำเข้าข้อมูลแล้ว เราจะมี DataFrame หลัก 2 ชุด 
- `farmer_raw_df` คือข้อมูลเกษตรกรเปราะบางจากหลาย sheet ในไฟล์ Excel 
- `mso_df` คือข้อมูล logbook จาก MSO 

ก่อนนำข้อมูลไป query ควรตรวจสอบข้อมูลเบื้องต้นก่อน เช่น 
- จำนวนแถวและจำนวนคอลัมน์ 
- ชื่อ column 
- ชนิดข้อมูล 
- ตัวอย่างข้อมูล 
- ค่า missing ใน column สำคัญ 

เพื่อตรวจสอบว่า DataFrame พร้อมสำหรับ JOIN ด้วย SQL หรือไม่

In [17]:
farmer_raw_df.shape

(95, 16)

In [18]:
farmer_raw_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


In [19]:
farmer_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   department_code  95 non-null     str    
 1   department       95 non-null     str    
 2   pid              95 non-null     string 
 3   province_code    67 non-null     string 
 4   province         10 non-null     object 
 5   amphur           10 non-null     object 
 6   tambon           10 non-null     object 
 7   is_farmer        95 non-null     int64  
 8   farmer_type      95 non-null     str    
 9   main_occupation  21 non-null     object 
 10  income_in        64 non-null     float64
 11  income_out       64 non-null     float64
 12  debts_in         64 non-null     float64
 13  debts_out        64 non-null     float64
 14  updated_at       80 non-null     float64
 15  source_sheet     95 non-null     str    
dtypes: float64(5), int64(1), object(4), str(4), string(2)
memory usage: 12.0+ K

In [20]:
mso_df.shape

(100, 14)

In [21]:
mso_df.head()

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,ตำบล/แขวง,เขต/อำเภอ/เทศบาล,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


In [22]:
mso_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   วันที่แก้ไขข้อมูลล่าสุด  100 non-null    datetime64[us]
 1   รหัสครัวเรือน            100 non-null    string        
 2   รหัสประจำบ้าน            24 non-null     string        
 3   วันที่สร้างครัวเรือน     100 non-null    datetime64[us]
 4   อายุ                     100 non-null    int64         
 5   เพศ                      77 non-null     str           
 6   ตำบล/แขวง                100 non-null    str           
 7   เขต/อำเภอ/เทศบาล         100 non-null    str           
 8   จังหวัด                  100 non-null    str           
 9   ระดับการศึกษา            77 non-null     str           
 10  อาชีพหลัก                39 non-null     str           
 11  ประเภทกลุ่มเป้าหมาย      100 non-null    str           
 12  รหัส cm                  100 non-null    string 

จากข้อมูลที่นำเข้า มี column สำคัญที่เกี่ยวข้องกับการเชื่อมข้อมูลดังนี้ 

### ข้อมูลเกษตรกร

| column | ความหมาย | 
|---|---| 
| department_code | รหัสหน่วยงาน | 
| department | หน่วยงาน | 
| pid | รหัสบุคคล | 
| province_code | รหัสจังหวัด | 
| province | จังหวัด | 
| amphur | อำเภอ | 
| tambon | ตำบล | 
| is_farmer | สถานะว่าเป็นเกษตรกรหรือไม่ | 
| farmer_type | ประเภทเกษตรกร | 
| main_occupation | อาชีพหลัก | 
| income_in | รายได้ในภาคเกษตร | 
| income_out | รายได้นอกภาคเกษตร | 
| debts_in | หนี้สินในภาคเกษตร | 
| debts_out | หนี้สินนอกภาคเกษตร | 
| updated_at | วันที่ปรับปรุงข้อมูล | 
| source_sheet | sheet ต้นทางของข้อมูล |

### ข้อมูล MSO 
| column | ความหมาย | 
|---|---| 
| วันที่แก้ไขข้อมูลล่าสุด | วันที่แก้ไขข้อมูลล่าสุด | 
| รหัสครัวเรือน | รหัสครัวเรือน | 
| รหัสประจำบ้าน | รหัสประจำบ้าน | 
| วันที่สร้างครัวเรือน | วันที่สร้างครัวเรือน | 
| ตำบล/แขวง | ตำบล | 
| เขต/อำเภอ/เทศบาล | อำเภอ | 
| จังหวัด | จังหวัด | 
| อายุ | อายุ | 
| เพศ | เพศ | 
| ระดับการศึกษา | ระดับการศึกษา | 
| อาชีพหลัก | อาชีพหลัก | 
| ประเภทกลุ่มเป้าหมาย | ประเภทกลุ่มเป้าหมาย | 
| รหัส cm | รหัส case manager | 
| หน่วยงานของ cm | หน่วยงานของ case manager |

## 3.5 เตรียมข้อมูลสำหรับใช้ Query 
เพื่อให้เขียน SQL ได้ง่ายขึ้น เราจะเตรียม DataFrame สำหรับใช้ในบทนี้ โดย 
1. คัดลอก `farmer_raw_df` มาเป็น `farmer_df` 
2. เปลี่ยนชื่อ column พื้นที่ของ `mso_df` ให้ตรงกับฝั่งเกษตรกร 
3. ทำความสะอาด key ที่จะใช้เชื่อมข้อมูล เช่น จังหวัด อำเภอ ตำบล 
4. สร้าง column เพิ่มเติมที่ช่วยให้การวิเคราะห์ง่ายขึ้น 

เหตุผลที่ต้องทำให้ชื่อ column พื้นที่ตรงกัน เพราะในขั้นตอน JOIN เราจะใช้พื้นที่เป็น key หลัก เช่น 
- `province` 
- `amphur` 
- `tambon` 

ถ้าชื่อ column ไม่ตรงกัน query จะอ่านยากและมีโอกาสเขียนผิดง่าย

In [23]:
farmer_df = farmer_raw_df.copy()

mso_df = mso_df.rename(columns={ 
    "จังหวัด": "province", 
    "เขต/อำเภอ/เทศบาล": "amphur", 
    "ตำบล/แขวง": "tambon", 
    "วันที่แก้ไขข้อมูลล่าสุด": "last_updated_at", 
    "วันที่สร้างครัวเรือน": "household_created_at", 
    "รหัสครัวเรือน": "household_id", 
    "รหัสประจำบ้าน": "house_code", 
    "อายุ": "age", 
    "เพศ": "gender", 
    "ระดับการศึกษา": "education_level", 
    "อาชีพหลัก": "main_occupation", 
    "ประเภทกลุ่มเป้าหมาย": "target_group", 
    "รหัส cm": "cm_id", 
    "หน่วยงานของ cm": "cm_department" 
})

In [25]:
list(farmer_df.columns)

['department_code',
 'department',
 'pid',
 'province_code',
 'province',
 'amphur',
 'tambon',
 'is_farmer',
 'farmer_type',
 'main_occupation',
 'income_in',
 'income_out',
 'debts_in',
 'debts_out',
 'updated_at',
 'source_sheet']

In [26]:
list(mso_df.columns)

['last_updated_at',
 'household_id',
 'house_code',
 'household_created_at',
 'age',
 'gender',
 'tambon',
 'amphur',
 'province',
 'education_level',
 'main_occupation',
 'target_group',
 'cm_id',
 'cm_department']

ตรวจสอบ column พื้นที่หลังปรับชื่อ

In [33]:
mso_df[["province", "amphur", "tambon"]]

,province,amphur,tambon
0,อุดรธานี,นายูง,โนนทอง
1,ราชบุรี,ดำเนินสะดวก,ท่านัด
2,หนองคาย,ท่าบ่อ,นาข่า
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว
...,...,...,...
95,ชัยภูมิ,เกษตรสมบูรณ์,โนนกอก
96,มหาสารคาม,พยัคฆภูมิพิสัย,เมืองเตา
97,สระแก้ว,วัฒนานคร,หนองหมากฝ้าย
98,ขอนแก่น,ภูเวียง,หนองกุงธนสาร


In [34]:
farmer_df[["province", "amphur", "tambon"]]

,province,amphur,tambon
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
...,...,...,...
90,NaN,NaN,NaN
91,NaN,NaN,NaN
92,NaN,NaN,NaN
93,NaN,NaN,NaN


## 3.6 ทำความสะอาด Key สำหรับการ JOIN 
ในบทนี้ เราจะใช้ชื่อพื้นที่เป็น key สำหรับเชื่อมข้อมูล ได้แก่ 
- `province` 
- `amphur` 
- `tambon` 

เนื่องจาก key เหล่านี้เป็นข้อความ จึงอาจมีปัญหา เช่น 
- มีช่องว่างหัวท้าย 
- มีค่าขาดหาย 
- ใช้คำสะกดไม่ตรงกัน 
- มีชื่ออำเภอหรือตำบลในรูปแบบที่ต่างกัน 

ก่อน JOIN จึงควรทำความสะอาดเบื้องต้น โดยแปลงเป็น string และตัดช่องว่างหัวท้าย

In [38]:
area_cols = ["province", "amphur", "tambon"] 

for col in area_cols: 
    farmer_df[col] = farmer_df[col].astype("string").str.strip() 
    mso_df[col] = mso_df[col].astype("string").str.strip()

ตรวจสอบ missing value ใน key ที่จะใช้เชื่อมข้อมูล

In [39]:
farmer_df[area_cols].isna().sum()

province    85
amphur      85
tambon      85
dtype: int64

In [40]:
mso_df[area_cols].isna().sum()

province    0
amphur      0
tambon      0
dtype: int64

ตรวจสอบจำนวนพื้นที่ที่ไม่ซ้ำในแต่ละชุดข้อมูล 

ขั้นตอนนี้ช่วยให้เห็นว่าข้อมูลแต่ละชุดครอบคลุมพื้นที่มากน้อยเพียงใด

In [41]:
farmer_df[area_cols].drop_duplicates().shape

(11, 3)

In [42]:
mso_df[area_cols].drop_duplicates().shape

(99, 3)

## 3.7 สร้าง Column สำหรับใช้วิเคราะห์เพิ่มเติม 
ข้อมูลเกษตรกรมีรายได้และหนี้สินแยกเป็นในภาคเกษตรและนอกภาคเกษตร เช่น 
- `income_in` 
- `income_out` 
- `debts_in` 
- `debts_out` 

เพื่อให้วิเคราะห์ง่ายขึ้น เราจะสร้าง column รวม ได้แก่ 
- `total_income` 
- `total_debt` 

column เหล่านี้จะช่วยให้ query วิเคราะห์รายได้และหนี้สินในระดับพื้นที่ได้ง่ายขึ้น

In [43]:
income_cols = ["income_in", "income_out"] 
debt_cols = ["debts_in", "debts_out"] 

for col in income_cols + debt_cols: 
    farmer_df[col] = pd.to_numeric(farmer_df[col], errors="coerce") 
    
farmer_df["total_income"] = farmer_df[income_cols].sum(axis=1, skipna=True) 
farmer_df["total_debt"] = farmer_df[debt_cols].sum(axis=1, skipna=True)

In [44]:
farmer_df[[ 
    "pid", 
    "province", 
    "amphur", 
    "tambon", 
    "income_in", 
    "income_out", 
    "total_income", 
    "debts_in", 
    "debts_out", 
    "total_debt" 
]].head()

,pid,province,amphur,tambon,income_in,income_out,total_income,debts_in,debts_out,total_debt
0,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
1,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
2,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
3,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
4,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0


สำหรับข้อมูล MSO เราสามารถใช้ข้อมูลประชากรและข้อมูลกลุ่มเป้าหมายในการวิเคราะห์ เช่น 
- อายุ 
- เพศ 
- ระดับการศึกษา 
- อาชีพหลัก 
- ประเภทกลุ่มเป้าหมาย 
- หน่วยงานของ CM 

ในบทนี้เราจะปรับชนิดข้อมูลของ `age` ให้เป็นตัวเลข เพื่อใช้วิเคราะห์ได้

In [45]:
mso_df["age"] = pd.to_numeric(mso_df["age"])

In [46]:
mso_df[[ 
    "household_id", 
    "house_code", 
    "province", 
    "amphur", 
    "tambon", 
    "age", 
    "gender", 
    "education_level", 
    "main_occupation", 
    "target_group", 
    "cm_department" 
]].head()

,household_id,house_code,province,amphur,tambon,age,gender,education_level,main_occupation,target_group,cm_department
0,648ff7d6b64d06b6b0dc7f02,<NA>,อุดรธานี,นายูง,โนนทอง,3,หญิง,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,62d3f7a6d6f101550054198b,70040223124,ราชบุรี,ดำเนินสะดวก,ท่านัด,55,หญิง,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,62a05c9df5674ecdd3805787,<NA>,หนองคาย,ท่าบ่อ,นาข่า,33,หญิง,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,62908e378fa67bf5ad7d5555,<NA>,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,5,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,6319b26ca3a37508384e36fc,<NA>,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,3,ชาย,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


## 3.8 สร้าง DuckDB Connection 
DuckDB จะทำหน้าที่เป็น SQL engine สำหรับ query pandas DataFrame 

ข้อดีคือ เราสามารถใช้ SQL กับ DataFrame ได้โดยตรง โดยไม่ต้องนำข้อมูลเข้า database แยกต่างหาก

In [47]:
con = duckdb.connect()

## 3.9 Register DataFrame เป็น Table 
ก่อน query ด้วย SQL ต้อง register DataFrame ให้ DuckDB รู้จักก่อน

ในบทนี้จะใช้ชื่อ table ว่า 
- `farmer` 
- `mso`

In [48]:
con.register("farmer", farmer_df) 
con.register("mso", mso_df)

In [52]:
con.sql(
    """ 
    SELECT * 
    FROM farmer 
    LIMIT 5 
    """
).df()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet,total_income,total_debt
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0


In [53]:
con.sql(
    """ 
    SELECT * 
    FROM mso 
    LIMIT 5 
    """
).df()

,last_updated_at,household_id,house_code,household_created_at,age,gender,tambon,amphur,province,education_level,main_occupation,target_group,cm_id,cm_department
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,NaN,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,NaN,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,NaN,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,NaN,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


## 3.10 SQL พื้นฐาน: SELECT 
`SELECT` ใช้เลือก column ที่ต้องการดูจากตาราง 

ตัวอย่าง: เลือกข้อมูลพื้นที่และหน่วยงานจากข้อมูลเกษตรกร

In [54]:
con.sql(
    """ 
    SELECT department, source_sheet, province, amphur, tambon, farmer_type, main_occupation 
    FROM farmer 
    LIMIT 10 
    """
).df()

,department,source_sheet,province,amphur,tambon,farmer_type,main_occupation
0,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
1,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
2,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
3,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
4,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
5,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
6,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
7,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
8,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
9,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None


ตัวอย่าง: เลือกข้อมูลพื้นที่และกลุ่มเป้าหมายจากข้อมูล MSO

In [55]:
con.sql(
    """ 
    SELECT province, amphur, tambon, age, gender, education_level, main_occupation, target_group, cm_department 
    FROM mso 
    LIMIT 10 
    """
).df()

,province,amphur,tambon,age,gender,education_level,main_occupation,target_group,cm_department
0,อุดรธานี,นายูง,โนนทอง,3,หญิง,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,ราชบุรี,ดำเนินสะดวก,ท่านัด,55,หญิง,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,หนองคาย,ท่าบ่อ,นาข่า,33,หญิง,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,5,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,3,ชาย,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
5,ประจวบคีรีขันธ์,เมืองประจวบคีรีขันธ์,อ่าวน้อย,39,ชาย,มัธยมศึกษาตอนต้น,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
6,อุดรธานี,เทศบาลนครอุดรธานี,หมากแข้ง,4,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
7,นครศรีธรรมราช,ปากพนัง,ปากพนังฝั่งตะวันออก,68,NaN,NaN,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,ผู้สูงอายุ,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.นครศรีธรรมราช
8,สระแก้ว,วัฒนานคร,วัฒนานคร,90,หญิง,ประถมศึกษา,NaN,ผู้สูงอายุ,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
9,บุรีรัมย์,หนองหงส์,ไทยสามัคคี,6,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็ก,บ้านพักเด็กและครอบครัวจังหวัดบุรีรัมย์


## 3.11 SQL พื้นฐาน: WHERE 
`WHERE` ใช้กรองข้อมูลตามเงื่อนไข 

ตัวอย่าง: เลือกข้อมูลเกษตรกรเฉพาะจังหวัดอุดรธานี